In [ ]:
# 基本ライブラリと使用モデルを読み込む
import os
import gc
import time
import warnings
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)


In [ ]:
# データ
class CFG:
    TARGET = "Churn"
    SEED = 42

    # 外側CVは最終的な汎化性能評価に使う
    N_SPLITS = 10
    # Target Encoding用の内側CV
    INNER_SPLITS = 5

    # 顧客セグメント数
    N_CLUSTERS = 6

    # カテゴリごとの平均が小標本に引っ張られすぎないように平滑化する
    TE_SMOOTH = 20

    USE_GPU = True

    TRAIN_PATHS = [
        "train.csv",
        "/kaggle/input/playground-series-s6e3/train.csv",
        "/kaggle/input/competitions/playground-series-s6e3/train.csv",
    ]
    TEST_PATHS = [
        "test.csv",
        "/kaggle/input/playground-series-s6e3/test.csv",
        "/kaggle/input/competitions/playground-series-s6e3/test.csv",
    ]
    SAMPLE_PATHS = [
        "sample_submission.csv",
        "/kaggle/input/playground-series-s6e3/sample_submission.csv",
        "/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv",
    ]
    ORIG_PATHS = [
        "WA_Fn-UseC_-Telco-Customer-Churn.csv",
        "/kaggle/input/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv",
        "/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    ]


# 元データのカテゴリ特徴量
RAW_CATS = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]

# 元データの数値特徴量
RAW_NUMS = ["tenure", "MonthlyCharges", "TotalCharges"]

# 組み合わせ特徴量を作る主要カテゴリ
TOP_CATS_FOR_NGRAM = [
    "Contract", "InternetService", "PaymentMethod",
    "OnlineSecurity", "TechSupport", "PaperlessBilling",
]


# 再現性を保つため乱数シードを固定する
def seed_everything(seed=42):
    np.random.seed(seed)


# ローカル実行とKaggle実行の両方に対応するため、存在するパスを選ぶ
def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No path exists in: {paths}")


seed_everything(CFG.SEED)


In [ ]:
# dataset読み込み
def load_data():
    train_path = first_existing(CFG.TRAIN_PATHS)
    test_path = first_existing(CFG.TEST_PATHS)
    orig_path = first_existing(CFG.ORIG_PATHS)

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    orig = pd.read_csv(orig_path)

    # sample_submissionは環境によって無い場合があるため、存在確認して読み込む
    sample = None
    for p in CFG.SAMPLE_PATHS:
        if os.path.exists(p):
            sample = pd.read_csv(p)
            break

    if "customerID" in orig.columns:
        orig = orig.drop(columns=["customerID"])

    # id列は提出ファイル作成時に必要なので先に保存しておく
    train_ids = train["id"].copy() if "id" in train.columns else pd.Series(np.arange(len(train)), name="id")
    test_ids = test["id"].copy() if "id" in test.columns else pd.Series(np.arange(len(test)), name="id")

    # 目的変数を0/1に変換する
    train[CFG.TARGET] = train[CFG.TARGET].map({"No": 0, "Yes": 1}).astype(int)
    if orig[CFG.TARGET].dtype == "object":
        orig[CFG.TARGET] = orig[CFG.TARGET].map({"No": 0, "Yes": 1}).astype(int)


    # TotalChargesは空白が混ざることがあるため数値型へ変換する
    for df in [train, test, orig]:
        if "TotalCharges" in df.columns:
            df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # 欠損値はtrainとoriginalの中央値で補完する
    tc_median = pd.concat([train["TotalCharges"], orig["TotalCharges"]], axis=0).median()
    for df in [train, test, orig]:
        df["TotalCharges"] = df["TotalCharges"].fillna(tc_median)

    print("Loaded data")
    print(f"train: {train.shape}, test: {test.shape}, original: {orig.shape}")
    return train, test, orig, sample, train_ids, test_ids


train, test, orig, sample, train_ids, test_ids = load_data()
display(train.head())


In [ ]:
# 特徴量生成
def add_domain_features(df):
    df = df.copy()

    df["charges_deviation"] = (df["TotalCharges"] - df["tenure"] * df["MonthlyCharges"]).astype("float32")
    df["monthly_to_total_ratio"] = (df["MonthlyCharges"] / (df["TotalCharges"] + 1)).astype("float32")
    df["avg_monthly_charges"] = (df["TotalCharges"] / (df["tenure"] + 1)).astype("float32")
    df["tenure_x_monthly"] = (df["tenure"] * df["MonthlyCharges"]).astype("float32")

    # 契約しているサービス数を集約し、利用状況の濃さを表す
    service_cols = [
        "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
        "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    ]
    df["service_count"] = (df[service_cols] == "Yes").sum(axis=1).astype("float32")
    df["security_count"] = (df[["OnlineSecurity", "TechSupport"]] == "Yes").sum(axis=1).astype("float32")
    df["streaming_count"] = (df[["StreamingTV", "StreamingMovies"]] == "Yes").sum(axis=1).astype("float32")
    df["has_internet"] = (df["InternetService"] != "No").astype("float32")
    df["has_phone"] = (df["PhoneService"] == "Yes").astype("float32")
    df["tenure_x_service"] = (df["tenure"] * df["service_count"]).astype("float32")


    # 契約期間や契約形態から、解約しやすさに関係しそうなフラグを作る
    df["is_new_customer"] = (df["tenure"] <= 1).astype("float32")
    df["is_long_contract"] = (df["Contract"] == "Two year").astype("float32")
    df["is_month_to_month"] = (df["Contract"] == "Month-to-month").astype("float32")

    return df


# 数値そのものだけでなく、丸め値・桁・端数などの表現を追加する
def add_digit_features(df):
    df = df.copy()

    for col, prefix, round_base in [
        ("tenure", "tenure", 10),
        ("MonthlyCharges", "mc", 10),
        ("TotalCharges", "tc", 100),
    ]:
        vals = df[col].fillna(0).astype(float)
        floored = np.floor(np.abs(vals)).astype(int)
        s = floored.astype(str)

        df[f"{prefix}_first_digit"] = s.str[0].astype("int16")
        df[f"{prefix}_last_digit"] = s.str[-1].astype("int16")
        df[f"{prefix}_num_digits"] = s.str.len().astype("int16")
        df[f"{prefix}_mod10"] = (floored % 10).astype("int16")
        df[f"{prefix}_mod100"] = (floored % 100).astype("int16")
        df[f"{prefix}_rounded"] = (np.round(vals / round_base) * round_base).astype("float32")
        df[f"{prefix}_dev_from_round"] = np.abs(vals - df[f"{prefix}_rounded"]).astype("float32")

    df["tenure_years"] = (df["tenure"] // 12).astype("int16")
    df["tenure_months_in_year"] = (df["tenure"] % 12).astype("int16")
    df["mc_fractional"] = (df["MonthlyCharges"] - np.floor(df["MonthlyCharges"])).astype("float32")
    df["tc_fractional"] = (df["TotalCharges"] - np.floor(df["TotalCharges"])).astype("float32")

    return df


# 値の出現頻度を特徴量化し、珍しいカテゴリ・値をモデルに伝える
def add_frequency_features(train, test, orig, cols):
    train, test, orig = train.copy(), test.copy(), orig.copy()
    for col in cols:
        ref = pd.concat([train[col], test[col], orig[col]], axis=0)
        freq = ref.value_counts(normalize=True, dropna=False)
        name = f"FREQ_{col}"
        for df in [train, test, orig]:
            df[name] = df[col].map(freq).fillna(0).astype("float32")
    return train, test, orig


# original datasetの目的変数平均を使い、カテゴリごとの解約率傾向を追加する
def add_original_target_stats(train, test, orig, cols):
    train, test = train.copy(), test.copy()
    global_mean = orig[CFG.TARGET].mean()

    for col in cols:
        stats = orig.groupby(col, dropna=False)[CFG.TARGET].agg(["mean", "count"]).reset_index()
        # smoothingにより、件数が少ないカテゴリの平均が極端になりすぎるのを防ぐ
        stats[f"ORIG_TE_{col}"] = (
            (stats["mean"] * stats["count"] + global_mean * CFG.TE_SMOOTH)
            / (stats["count"] + CFG.TE_SMOOTH)
        ).astype("float32")
        stats[f"ORIG_COUNT_{col}"] = np.log1p(stats["count"]).astype("float32")

        keep = [col, f"ORIG_TE_{col}", f"ORIG_COUNT_{col}"]
        train = train.merge(stats[keep], on=col, how="left")
        test = test.merge(stats[keep], on=col, how="left")

        for df in [train, test]:
            df[f"ORIG_TE_{col}"] = df[f"ORIG_TE_{col}"].fillna(global_mean).astype("float32")
            df[f"ORIG_COUNT_{col}"] = df[f"ORIG_COUNT_{col}"].fillna(0).astype("float32")

    return train, test


# Churn / Non-Churnそれぞれの分布と比べた位置関係を特徴量化する
def add_distribution_features(train, test, orig):
    train, test = train.copy(), test.copy()

    def pctrank_against(values, reference):
        reference = np.sort(np.asarray(reference, dtype=float))
        if len(reference) == 0:
            return np.zeros(len(values), dtype="float32")
        return (np.searchsorted(reference, values) / len(reference)).astype("float32")

    def zscore_against(values, reference):
        reference = np.asarray(reference, dtype=float)
        mu, sigma = np.mean(reference), np.std(reference)
        if sigma == 0:
            return np.zeros(len(values), dtype="float32")
        return ((values - mu) / sigma).astype("float32")

    # original datasetを基準に、解約者・非解約者のTotalCharges分布を作る
    churn_tc = orig.loc[orig[CFG.TARGET] == 1, "TotalCharges"].values
    nonchurn_tc = orig.loc[orig[CFG.TARGET] == 0, "TotalCharges"].values
    all_tc = orig["TotalCharges"].values

    internet_mc_mean = orig.groupby("InternetService")["MonthlyCharges"].mean()

    for df in [train, test]:
        tc = df["TotalCharges"].values.astype(float)

        # TotalChargesが各分布の中でどの位置にあるかを表す
        df["pctrank_churner_TC"] = pctrank_against(tc, churn_tc)
        df["pctrank_nonchurner_TC"] = pctrank_against(tc, nonchurn_tc)
        df["pctrank_orig_TC"] = pctrank_against(tc, all_tc)
        df["pctrank_churn_gap_TC"] = (df["pctrank_churner_TC"] - df["pctrank_nonchurner_TC"]).astype("float32")

        df["zscore_churn_TC"] = zscore_against(tc, churn_tc)
        df["zscore_nonchurn_TC"] = zscore_against(tc, nonchurn_tc)
        df["zscore_churn_gap_TC"] = (
            np.abs(df["zscore_churn_TC"]) - np.abs(df["zscore_nonchurn_TC"])
        ).astype("float32")

        # InternetService別の平均月額料金からのズレを表す
        df["resid_IS_MC"] = (
            df["MonthlyCharges"] - df["InternetService"].map(internet_mc_mean).fillna(orig["MonthlyCharges"].mean())
        ).astype("float32")

        for q_label, q in [("q25", 0.25), ("q50", 0.50), ("q75", 0.75)]:
            ch_q = np.quantile(churn_tc, q)
            nc_q = np.quantile(nonchurn_tc, q)
            df[f"dist_churn_TC_{q_label}"] = np.abs(df["TotalCharges"] - ch_q).astype("float32")
            df[f"dist_nonchurn_TC_{q_label}"] = np.abs(df["TotalCharges"] - nc_q).astype("float32")
            df[f"qdist_gap_TC_{q_label}"] = (
                df[f"dist_nonchurn_TC_{q_label}"] - df[f"dist_churn_TC_{q_label}"]
            ).astype("float32")

    return train, test


# 重要そうなカテゴリ同士を組み合わせ、単独列では表せない相互作用を作る
def add_composite_categoricals(train, test, top_cols):
    train, test = train.copy(), test.copy()
    new_cols = []

    for c1, c2 in combinations(top_cols, 2):
        name = f"BG_{c1}_{c2}"
        for df in [train, test]:
            df[name] = (df[c1].astype(str) + "__" + df[c2].astype(str)).astype("category")
        new_cols.append(name)

    for c1, c2, c3 in combinations(top_cols[:4], 3):
        name = f"TG_{c1}_{c2}_{c3}"
        for df in [train, test]:
            df[name] = (
                df[c1].astype(str) + "__" + df[c2].astype(str) + "__" + df[c3].astype(str)
            ).astype("category")
        new_cols.append(name)

    return train, test, new_cols


In [ ]:
# 実行
def build_features(train, test, orig):
    train, test, orig = train.copy(), test.copy(), orig.copy()

    # 1. ドメイン特徴
    train = add_domain_features(train)
    test = add_domain_features(test)
    orig = add_domain_features(orig)

    # 2. 数字の丸め・桁特徴
    train = add_digit_features(train)
    test = add_digit_features(test)
    orig = add_digit_features(orig)

    # 3. 頻度特徴
    freq_cols = RAW_CATS + RAW_NUMS
    train, test, orig = add_frequency_features(train, test, orig, freq_cols)

    # 4. original dataset由来の目的変数統計
    stat_cols = RAW_CATS + RAW_NUMS
    train, test = add_original_target_stats(train, test, orig, stat_cols)

    # 5. Churn / Non-Churn 分布との距離
    train, test = add_distribution_features(train, test, orig)

    # 6. 数値をカテゴリ化
    num_as_cat = []
    for col in RAW_NUMS:
        name = f"CAT_{col}"
        train[name] = train[col].astype(str).astype("category")
        test[name] = test[col].astype(str).astype("category")
        num_as_cat.append(name)

    # 7. カテゴリN-gram（カテゴリ同士の組み合わせ特徴）
    train, test, ngram_cols = add_composite_categoricals(train, test, TOP_CATS_FOR_NGRAM)

    cat_cols = RAW_CATS + num_as_cat + ngram_cols

    # モデルに渡す特徴量リストを作り、カテゴリ列の型をそろえる
    drop_cols = [CFG.TARGET]
    if "id" in train.columns:
        drop_cols.append("id")
    if "id" in test.columns:
        test = test.drop(columns=["id"])

    features = [c for c in train.columns if c not in drop_cols]
    for c in cat_cols:
        if c in features:
            train[c] = train[c].astype(str).fillna("Missing").astype("category")
            test[c] = test[c].astype(str).fillna("Missing").astype("category")

    print(f"Total features: {len(features)}")
    print(f"Categorical features: {len(cat_cols)}")
    return train, test, features, cat_cols


# 特徴量生成を実行し、以降の学習で使う列を確定する
train_fe, test_fe, FEATURES, CAT_COLS = build_features(train, test, orig)
display(train_fe[FEATURES + [CFG.TARGET]].head())


In [ ]:
# KMeansで顧客をクラスタリングし、セグメント情報を特徴量として追加
def add_segment_features(X_tr, X_va, X_te, cat_cols):
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()

    # クラスタリングには、契約期間・料金・サービス数などの基本特徴を使う
    base_num = [c for c in RAW_NUMS + [
        "charges_deviation", "monthly_to_total_ratio", "avg_monthly_charges",
        "service_count", "security_count", "streaming_count",
        "tenure_x_monthly", "resid_IS_MC"
    ] if c in X_tr.columns]

    base_cat = [c for c in ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport"] if c in X_tr.columns]

    # 数値は標準化、カテゴリはOne-Hot化してKMeansに入力する
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), base_num),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), base_cat),
        ],
        remainder="drop",
    )

    Xt = pre.fit_transform(X_tr)
    Xv = pre.transform(X_va)
    Xs = pre.transform(X_te)

    # foldの学習データだけでクラスタを作り、validation/testへ適用する
    km = KMeans(n_clusters=CFG.N_CLUSTERS, random_state=CFG.SEED, n_init=10)
    X_tr["risk_cluster"] = km.fit_predict(Xt).astype(str)
    X_va["risk_cluster"] = km.predict(Xv).astype(str)
    X_te["risk_cluster"] = km.predict(Xs).astype(str)

    # 最近傍クラスタ中心までの距離も、セグメントへの当てはまり度として使う
    X_tr["risk_cluster_dist"] = km.transform(Xt).min(axis=1).astype("float32")
    X_va["risk_cluster_dist"] = km.transform(Xv).min(axis=1).astype("float32")
    X_te["risk_cluster_dist"] = km.transform(Xs).min(axis=1).astype("float32")

    for df in [X_tr, X_va, X_te]:
        df["risk_cluster"] = df["risk_cluster"].astype("category")

    cat_cols = cat_cols + ["risk_cluster"]
    return X_tr, X_va, X_te, cat_cols


# fold内でTarget Encodingを行い、validationへのリークを避ける
def add_cv_target_encoding(X_tr, y_tr, X_va, X_te, cat_cols):
    """fold内でさらにinner CVを使うTarget Encoding。train側のリークを防ぐ。"""
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()
    y_tr = pd.Series(y_tr).reset_index(drop=True)
    global_mean = y_tr.mean()

    skf = StratifiedKFold(n_splits=CFG.INNER_SPLITS, shuffle=True, random_state=CFG.SEED)

    for col in cat_cols:
        col_as_str = X_tr[col].astype(str).reset_index(drop=True)
        te_name = f"CVTE_{col}"
        cnt_name = f"CVCNT_{col}"

        oof_te = np.zeros(len(X_tr), dtype="float32")
        oof_cnt = np.zeros(len(X_tr), dtype="float32")

        tmp_base = pd.DataFrame({"key": col_as_str, "y": y_tr})

        # train側はinner CVでOOF形式のTarget Encodingを作る
        for in_idx, hold_idx in skf.split(X_tr, y_tr):
            stats = tmp_base.iloc[in_idx].groupby("key")["y"].agg(["mean", "count"])
            smooth = (stats["mean"] * stats["count"] + global_mean * CFG.TE_SMOOTH) / (stats["count"] + CFG.TE_SMOOTH)

            hold_keys = col_as_str.iloc[hold_idx]
            oof_te[hold_idx] = hold_keys.map(smooth).fillna(global_mean).astype("float32").values
            oof_cnt[hold_idx] = np.log1p(hold_keys.map(stats["count"]).fillna(0)).astype("float32").values

        # validation/testにはfold学習データ全体から作った統計量を適用する
        full_stats = tmp_base.groupby("key")["y"].agg(["mean", "count"])
        full_smooth = (full_stats["mean"] * full_stats["count"] + global_mean * CFG.TE_SMOOTH) / (
            full_stats["count"] + CFG.TE_SMOOTH
        )

        X_tr[te_name] = oof_te
        X_tr[cnt_name] = oof_cnt

        for df in [X_va, X_te]:
            keys = df[col].astype(str)
            df[te_name] = keys.map(full_smooth).fillna(global_mean).astype("float32")
            df[cnt_name] = np.log1p(keys.map(full_stats["count"]).fillna(0)).astype("float32")

    return X_tr, X_va, X_te


# 数値欠損はfoldの学習データの中央値で補完し、カテゴリ型も再調整する
def fill_numeric_by_fold(X_tr, X_va, X_te, cat_cols):
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()
    num_cols = [c for c in X_tr.columns if c not in cat_cols]

    med = X_tr[num_cols].median(numeric_only=True)
    for df in [X_tr, X_va, X_te]:
        df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
        df[num_cols] = df[num_cols].fillna(med).fillna(0)

    for c in cat_cols:
        for df in [X_tr, X_va, X_te]:
            df[c] = df[c].astype(str).fillna("Missing").astype("category")

    return X_tr, X_va, X_te


In [ ]:
# ベースモデルをまとめて定義する。木系3モデルに線形モデルを加えて多様性を持たせる
def make_models():
    xgb_device = "cuda" if CFG.USE_GPU else "cpu"
    cb_task = "GPU" if CFG.USE_GPU else "CPU"

    return {
        # セグメント特徴を含めたXGBoost
        "xgb_segment": XGBClassifier(
            n_estimators=30000,
            learning_rate=0.01,
            max_depth=5,
            subsample=0.85,
            colsample_bytree=0.45,
            min_child_weight=5,
            reg_alpha=2.0,
            reg_lambda=2.0,
            gamma=0.2,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            enable_categorical=True,
            early_stopping_rounds=500,
            random_state=CFG.SEED,
            device=xgb_device,
            verbosity=0,
        ),

        # カテゴリ特徴を扱いやすいCatBoost
        "cat_segment": CatBoostClassifier(
            iterations=30000,
            learning_rate=0.03,
            depth=5,
            l2_leaf_reg=3.0,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=CFG.SEED,
            early_stopping_rounds=300,
            task_type=cb_task,
            verbose=False,
        ),

        # 勾配ブースティング系の別実装としてLightGBMも併用する
        "lgbm_segment": LGBMClassifier(
            n_estimators=30000,
            learning_rate=0.01,
            num_leaves=64,
            max_depth=6,
            min_child_samples=80,
            subsample=0.8,
            colsample_bytree=0.55,
            reg_alpha=1.0,
            reg_lambda=1.0,
            objective="binary",
            metric="auc",
            random_state=CFG.SEED,
            verbose=-1,
        ),

        # 非線形モデルだけに寄らないよう、One-Hot + Logistic Regressionも入れる
        "linear_ohe": None,
    }


# モデル名に応じてfit/predict処理を切り替える
def fit_predict_one_model(model_name, model, X_tr, y_tr, X_va, y_va, X_te, cat_cols):
    if model_name == "linear_ohe":
        # 線形モデルではカテゴリをOne-Hot化し、数値は標準化する
        num_cols = [c for c in X_tr.columns if c not in cat_cols]
        pre = ColumnTransformer(
            transformers=[
                ("num", Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]), num_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), cat_cols),
            ],
            remainder="drop",
        )
        model = Pipeline([
            ("pre", pre),
            ("lr", LogisticRegression(C=0.5, max_iter=5000, solver="lbfgs"))
        ])
        model.fit(X_tr, y_tr)
        va_pred = model.predict_proba(X_va)[:, 1]
        te_pred = model.predict_proba(X_te)[:, 1]
        return va_pred, te_pred

    # 以下は各ライブラリのfit方法に合わせて学習・予測する
    if model_name.startswith("xgb"):
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]

    if model_name.startswith("cat"):
        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_cols,
            verbose=False,
        )
        return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]

    if model_name.startswith("lgbm"):
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            categorical_feature=cat_cols,
            callbacks=[early_stopping(300, verbose=False), log_evaluation(0)],
        )
        return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]

    raise ValueError(f"Unknown model: {model_name}")


In [ ]:
# 外側CVで各ベースモデルのOOF予測とtest予測を作成する
y = train_fe[CFG.TARGET].values
X_all = train_fe[FEATURES].copy()
X_test_all = test_fe[FEATURES].copy()

models_template = make_models()
model_names = list(models_template.keys())

# OOF予測はstackingのメタ特徴量として使う

oof = {name: np.zeros(len(X_all), dtype="float32") for name in model_names}
pred = {name: np.zeros(len(X_test_all), dtype="float32") for name in model_names}
fold_scores = {name: [] for name in model_names}

skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

t0 = time.time()

# 各foldで特徴量生成も学習データ内に閉じて行い、リークを防ぐ
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y), 1):
    print(f"\n========== Fold {fold}/{CFG.N_SPLITS} ==========")

    X_tr = X_all.iloc[tr_idx].reset_index(drop=True).copy()
    y_tr = y[tr_idx]
    X_va = X_all.iloc[va_idx].reset_index(drop=True).copy()
    y_va = y[va_idx]
    X_te = X_test_all.reset_index(drop=True).copy()

    fold_cat_cols = [c for c in CAT_COLS if c in X_tr.columns]

    # 1. セグメント特徴
    X_tr, X_va, X_te, fold_cat_cols = add_segment_features(X_tr, X_va, X_te, fold_cat_cols)

    # 2. fold内 Target Encoding
    X_tr, X_va, X_te = add_cv_target_encoding(X_tr, y_tr, X_va, X_te, fold_cat_cols)

    # 3. 数値欠損補完
    X_tr, X_va, X_te = fill_numeric_by_fold(X_tr, X_va, X_te, fold_cat_cols)

    # 同じfoldのデータで複数モデルを学習し、予測を保存する
    for name in model_names:
        print(f"  training {name}...")
        model = make_models()[name]
        va_pred, te_pred = fit_predict_one_model(
            name, model, X_tr, y_tr, X_va, y_va, X_te, fold_cat_cols
        )

        oof[name][va_idx] = va_pred
        # test予測はfold平均を最終予測として使う
        pred[name] += te_pred / CFG.N_SPLITS

        auc = roc_auc_score(y_va, va_pred)
        fold_scores[name].append(auc)
        print(f"    AUC: {auc:.6f}")

        del model
        gc.collect()

    print(f"  elapsed: {(time.time() - t0) / 60:.1f} min")

print("\nBase model CV scores")
for name in model_names:
    print(f"{name:15s} OOF AUC={roc_auc_score(y, oof[name]):.6f} | mean={np.mean(fold_scores[name]):.6f} ± {np.std(fold_scores[name]):.6f}")


In [ ]:
# 予測確率をlogit変換し、メタモデルが確率差を扱いやすくする
def safe_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


# ベースモデルの予測値から、stacking用のメタ特徴量を作る
def make_meta_features(oof_dict, pred_dict):
    meta_train = pd.DataFrame({f"pred_{k}": v for k, v in oof_dict.items()})
    meta_test = pd.DataFrame({f"pred_{k}": v for k, v in pred_dict.items()})

    base_cols = list(meta_train.columns)

    # 予測の一致度・不確実性
    for df in [meta_train, meta_test]:
        df["pred_mean"] = df[base_cols].mean(axis=1)
        df["pred_std"] = df[base_cols].std(axis=1)
        df["pred_min"] = df[base_cols].min(axis=1)
        df["pred_max"] = df[base_cols].max(axis=1)
        df["pred_range"] = df["pred_max"] - df["pred_min"]

        for c in base_cols:
            df[f"logit_{c}"] = safe_logit(df[c].values)

    return meta_train, meta_test


# OOF予測を使ってメタモデルを学習し、最終予測を作る
meta_train, meta_test = make_meta_features(oof, pred)
display(meta_train.head())

meta_oof = np.zeros(len(meta_train), dtype="float32")
meta_pred = np.zeros(len(meta_test), dtype="float32")

meta_cv = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED + 777)

# メタモデルもCVで評価し、過学習していないか確認する
for fold, (tr_idx, va_idx) in enumerate(meta_cv.split(meta_train, y), 1):
    meta_model = LogisticRegression(C=0.3, max_iter=5000, solver="lbfgs")
    meta_model.fit(meta_train.iloc[tr_idx], y[tr_idx])

    meta_oof[va_idx] = meta_model.predict_proba(meta_train.iloc[va_idx])[:, 1]
    meta_pred += meta_model.predict_proba(meta_test)[:, 1] / CFG.N_SPLITS

    print(f"Meta Fold {fold}: AUC={roc_auc_score(y[va_idx], meta_oof[va_idx]):.6f}")

print("\n" + "=" * 60)
print(f"Final Segment-aware Stacking OOF AUC: {roc_auc_score(y, meta_oof):.6f}")
print("=" * 60)


In [ ]:
# OOF予測と提出用ファイルを保存する
oof_df = pd.DataFrame({"id": train_ids, CFG.TARGET: meta_oof})
for name in model_names:
    oof_df[f"{CFG.TARGET}_{name}"] = oof[name]
oof_df.to_csv("oof_segment_stacking.csv", index=False)

# sample_submissionがある場合はその形式に合わせて提出ファイルを作る
if sample is not None and "id" in sample.columns:
    submission = sample.copy()
    submission[CFG.TARGET] = meta_pred
else:
    submission = pd.DataFrame({"id": test_ids, CFG.TARGET: meta_pred})

submission.to_csv("submission.csv", index=False)

print("Saved:")
print("- oof_segment_stacking.csv")
print("- submission.csv")
display(submission.head())
